# Quantization: Qwen3-8B fp16 vs AWQ 4-bit

Runs on **Kaggle, GPU T4 x2, Internet ON**, with the `civil-code-corpus` dataset added (same as `kaggle_gpu_eval.ipynb`).

Both servers can't fit on the GPUs at once, so it runs one at a time:

1. **AWQ** (`Qwen/Qwen3-8B-AWQ`) served: answer all 54 questions, timed
2. **fp16** (`Qwen/Qwen3-8B`) served: answer the same 54, timed, then **judge both sets with the fp16 model**
3. Compare: RAGAS (faithfulness drop must be < 0.03), latency p50/p95, time to first token, throughput, weight memory. Logged to MLflow, written to `reports/quantization.{json,md}`.

Long steps write to log files and print the end: the Kaggle display sometimes freezes on long outputs. If it looks stuck, reload the page (F5); the work keeps running. Budget: about 1-1.5 GPU-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and corpus

In [ ]:
!git clone -q https://github.com/ZeinabMahfouz/The-Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1
!mkdir -p data/interim eval_out && cp "$(find /kaggle/input -name civil_code.json | head -1)" data/interim/
# The two md5 values must match (corpus = what dvc.lock describes)
!md5sum data/interim/civil_code.json
!grep -A3 "path: data/interim/civil_code.json" dvc.lock | grep "md5:"

## 2. vLLM in its own environment

Same recipe as the evaluation notebook: `uv` (Kaggle's `python -m venv` has no ensurepip), and the server started with `subprocess.Popen` (Kaggle doesn't allow `!cmd &`).

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
!/kaggle/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import os, signal, subprocess, time, urllib.request

def start_vllm(model, log_path):
    """Qwen3-8B (fp16 or AWQ) over both T4s. T4 has no bf16, so half precision.
    0.75 leaves room for BGE-M3 (indexing on cuda:1, RAGAS embeddings on cuda:0)."""
    log = open(log_path, "w")
    return subprocess.Popen(
        ["/kaggle/vllm-env/bin/vllm", "serve", model,
         "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "8192",
         "--gpu-memory-utilization", "0.75", "--port", "8000"],
        stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONPATH": ""}, start_new_session=True,
    )

def wait_ready(log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"vLLM ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- see the log:"); os.system(f"tail -n 30 {log_path}"); return False

def stop_vllm(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=120)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(10)
    os.system("nvidia-smi --query-gpu=index,memory.used --format=csv")  # should be near 0 again

## 3. Evaluation environment

In [ ]:
!pip install -q "ragas==0.4.3" "langchain-openai==1.1.9" "langchain-huggingface==1.2.2" \
    "qdrant-client==1.19.1" "mlflow==3.16.1" "openai==1.109.1" "instructor==1.3.2" \
    sentence-transformers pyyaml langfuse==4.15.6 prometheus-client==0.26.0
!pip install -q -e . --no-deps

## 4. AWQ 4-bit: serve, answer, stop

First start downloads ~6 GB. If the server fails with a quantization error, add `"--quantization", "awq"` to the command in `start_vllm` and re-run.

In [ ]:
awq = start_vllm("Qwen/Qwen3-8B-AWQ", "/kaggle/working/vllm_awq.log")
wait_ready("/kaggle/working/vllm_awq.log")
!grep -iE "quantiz|took" /kaggle/working/vllm_awq.log | tail -n 5

In [ ]:
# 54 questions one at a time (latency), then all at once with 8 threads (throughput). ~10 min.
!python scripts/quant_compare.py generate --label awq --model Qwen/Qwen3-8B-AWQ --embed-device cuda:1 \
    --vllm-log /kaggle/working/vllm_awq.log > eval_out/quant_awq.log 2>&1; tail -n 12 eval_out/quant_awq.log

In [ ]:
stop_vllm(awq)

## 5. fp16: serve, answer, then judge both answer sets

In [ ]:
fp16 = start_vllm("Qwen/Qwen3-8B", "/kaggle/working/vllm_fp16.log")
wait_ready("/kaggle/working/vllm_fp16.log")
!grep -iE "took" /kaggle/working/vllm_fp16.log | tail -n 3

In [ ]:
!python scripts/quant_compare.py generate --label fp16 --model Qwen/Qwen3-8B --embed-device cuda:1 \
    --vllm-log /kaggle/working/vllm_fp16.log > eval_out/quant_fp16.log 2>&1; tail -n 12 eval_out/quant_fp16.log

In [ ]:
# RAGAS on both sets, same fp16 judge. ~15-20 min.
!python scripts/quant_compare.py score --labels fp16 awq --judge-model Qwen/Qwen3-8B \
    > eval_out/quant_score.log 2>&1; grep "\[score\]" eval_out/quant_score.log

## 6. Compare and download

In [ ]:
!python scripts/quant_compare.py report > eval_out/quant_report.log 2>&1; cat reports/quantization.md

In [ ]:
!zip -qr /kaggle/working/quant_results.zip eval_out/quant eval_out/mlflow.db eval_out/quant_*.log \
    reports/quantization.json reports/quantization.md
!cp /kaggle/working/vllm_*.log /kaggle/working/repo/eval_out/ && cd /kaggle/working/repo && zip -qr /kaggle/working/quant_results.zip eval_out/vllm_awq.log eval_out/vllm_fp16.log
%cd /kaggle/working
from IPython.display import FileLink
display(FileLink("quant_results.zip"))